# 06 — Live Dashboard

*Spec section 10 (notebooks), driven by section 4.4-4.5 (live update behavior & staleness) and 3.6 (edge tracking).*

During the tournament the system locks each matchday's real results, re-estimates team
strengths (Bayesian posterior update), and re-simulates the remaining bracket. This notebook
is the analyst-facing companion to the Streamlit dashboard and demonstrates:

- **updated predictions** after results land (re-running `simulate_tournament` on a
  partially-completed tournament),
- **evolving team strengths** as the live update nudges attack/defense parameters,
- **edge tracking** over time (how flagged market edges move as the model updates),
- **model staleness** via `StalenessMonitor` (spec 4.5: cumulative-surprise yellow/red).

As with notebooks 04/05 it prefers real artifacts and ingested live results, falling back to
a small synthetic tournament-in-progress so the live narrative runs end to end.

## Imports

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from polymbappe.config import Settings
from polymbappe.simulate.structure import load_structure_2026
from polymbappe.simulate.tournament import (
    StalenessMonitor,
    StrengthModel,
    simulate_tournament,
    surprise_increment,
)
from polymbappe.eval.market import compute_edges

settings = Settings()
N_SIMS = 2_000  # production: 100_000 with --live (spec 4.4)
rng = np.random.default_rng(settings.random_seed)

## Structure and a baseline strength model

We load the 2026 structure and a pre-tournament strength model exactly as in notebook 05
(trained Dixon-Coles artifact when present, synthetic otherwise). This is the "as of kickoff"
state the live updates evolve from.

In [ ]:
structure = load_structure_2026(settings)
teams = structure.teams


def _baseline_model() -> StrengthModel:
    try:
        from polymbappe.features.context import HOSTS_2026
        from polymbappe.models.train import load_artifact

        dc = load_artifact("dixon_coles", settings)
        return StrengthModel.from_dixon_coles(dc, hosts=HOSTS_2026)
    except Exception as exc:  # pragma: no cover - notebook resilience
        print(f"No trained artifact ({exc!r}); synthesizing a StrengthModel.")
        base = structure.elo or {t: 1500.0 for t in teams}
        center = float(np.mean(list(base.values())))
        attack = {t: (base[t] - center) / 400.0 for t in teams}
        defense = {t: -(base[t] - center) / 400.0 for t in teams}
        return StrengthModel(attack=attack, defense=defense, home_advantage=0.25)


base_model = _baseline_model()
pre = simulate_tournament(structure, base_model, n_sims=N_SIMS, rng=rng).stage_probabilities()
print("Pre-tournament trophy favorites:")
pre.select(["team", "champion"]).head(8)

## Matchday results: real or synthetic

During the live tournament `polymbappe ingest --live` appends real results to the `matches`
table. Here we load completed 2026 World Cup matches if available; otherwise we synthesize a
first matchday — including a deliberate upset — so the live update and staleness logic have
something to react to. Each result carries the model's *pre-match* predicted probability for
the realized outcome, which is what staleness detection consumes (spec 4.5).

In [ ]:
def _match_hda(model: StrengthModel, home: str, away: str) -> tuple[float, float, float]:
    m = model.score_matrix(home, away, neutral=True)
    return (float(np.tril(m, -1).sum()), float(np.trace(m)), float(np.triu(m, 1).sum()))


def _load_results() -> tuple[pl.DataFrame, bool]:
    """Completed 2026 matches if ingested; else a synthetic first matchday with an upset."""
    try:
        from datetime import date
        from polymbappe.data.store import read_table, table_exists
        from polymbappe.data.tables import Table

        if table_exists(Table.MATCHES, settings):
            m = read_table(Table.MATCHES, settings)
            live = m.filter(
                (pl.col("competition") == "FIFA World Cup") & (pl.col("date") >= date(2026, 6, 11))
            )
            if not live.is_empty():
                return live, True
    except Exception as exc:  # pragma: no cover - notebook resilience
        print(f"Live results unavailable ({exc!r}); synthesizing a first matchday.")
    # Synthetic matchday 1: one match per group, favorite vs. underdog, with an upset in A.
    rows = []
    for gi, (group, mem) in enumerate(structure.groups.items()):
        home, away = mem[0], mem[1]
        if gi == 0:  # engineered upset: underdog wins
            hg, ag = 0, 2
        else:
            hg, ag = (2, 0) if rng.random() < 0.6 else (1, 1)
        rows.append({"match_id": f"{group}-0", "group": group,
                     "home_team": home, "away_team": away, "home_goals": hg, "away_goals": ag})
    return pl.DataFrame(rows), False


results, is_real_results = _load_results()
print(f"{'REAL' if is_real_results else 'SYNTHETIC'} live results: {results.height} matches")
results.head()

## Live strength re-estimation

Spec 4.4: after each matchday, re-estimate team strengths. We apply a lightweight Bayesian
posterior nudge analogous to the simulator's within-sim correlated update (spec 4.1):
`attack += lr * (observed_GD - expected_GD)` per team. Overperformers shift up, the model
updates, and we re-simulate the remaining bracket. (Production uses the full Bayesian
posterior update; this mirrors the mechanism for the notebook.)

In [ ]:
import copy

LR = 0.05  # small learning rate to avoid overreacting to single-match variance (spec 4.1)
live_model = copy.deepcopy(base_model)
updates: list[dict] = []

for r in results.iter_rows(named=True):
    h, a = r["home_team"], r["away_team"]
    lam, mu = base_model.rates(h, a, neutral=True)
    obs_gd = r["home_goals"] - r["away_goals"]
    exp_gd = lam - mu
    delta = LR * (obs_gd - exp_gd)
    live_model.attack[h] = live_model.attack.get(h, 0.0) + delta
    live_model.attack[a] = live_model.attack.get(a, 0.0) - delta
    updates.append({"team": h, "attack_delta": delta})
    updates.append({"team": a, "attack_delta": -delta})

strength_changes = (
    pl.DataFrame(updates).group_by("team").agg(pl.col("attack_delta").sum())
    .sort(pl.col("attack_delta").abs(), descending=True)
)
print("Largest live strength shifts:")
strength_changes.head(8)

## Updated predictions: before vs. after

Re-simulate with the updated strengths and compare trophy probabilities to the
pre-tournament baseline. The Reflect node (spec 5.2) flags teams whose probability moved by
more than 0.5pp — we surface the same shifts here.

In [ ]:
post = simulate_tournament(structure, live_model, n_sims=N_SIMS, rng=rng).stage_probabilities()

comparison = (
    pre.select(["team", pl.col("champion").alias("champion_pre")])
    .join(post.select(["team", pl.col("champion").alias("champion_post")]), on="team")
    .with_columns(((pl.col("champion_post") - pl.col("champion_pre")) * 100).alias("shift_pp"))
    .sort(pl.col("shift_pp").abs(), descending=True)
)
significant = comparison.filter(pl.col("shift_pp").abs() > 0.5)  # spec 5.2 threshold
print(f"{significant.height} teams shifted > 0.5pp (dashboard-notification threshold)")

show = comparison.head(10)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["seagreen" if v >= 0 else "crimson" for v in show["shift_pp"]]
ax.barh(show["team"][::-1], show["shift_pp"][::-1], color=colors[::-1])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("trophy probability shift (pp)")
ax.set_title("Live update: post-matchday vs. pre-tournament")
plt.tight_layout()
plt.show()
comparison.head(10)

## Edge tracking

As the model updates, flagged market edges move. We build post-update H/D/A predictions for
the *remaining* (not-yet-played) group matches and recompute edges against the market (spec
3.6). Real odds come from the `market_odds` table; absent that we synthesize a noisy market
to demonstrate the tracking workflow.

In [ ]:
_PAIRS = [(0, 1), (2, 3), (0, 2), (1, 3), (0, 3), (1, 2)]
played = set(results["match_id"].to_list())

rows = []
for group, mem in structure.groups.items():
    for k, (i, j) in enumerate(_PAIRS):
        mid = f"{group}-{k}"
        if mid in played:
            continue
        ph, pd, pa = _match_hda(live_model, mem[i], mem[j])
        rows.append({"match_id": mid, "home_team": mem[i], "away_team": mem[j],
                     "model_home": ph, "model_draw": pd, "model_away": pa})
remaining = pl.DataFrame(rows)


def _market(ids: list[str], preds: pl.DataFrame) -> tuple[pl.DataFrame, bool]:
    try:
        from polymbappe.data.store import read_table, table_exists
        from polymbappe.data.tables import Table

        if table_exists(Table.MARKET_ODDS, settings):
            return read_table(Table.MARKET_ODDS, settings), True
    except Exception as exc:  # pragma: no cover - notebook resilience
        print(f"Market odds unavailable ({exc!r}); synthesizing.")
    base = preds.select(["model_home", "model_draw", "model_away"]).to_numpy()
    p = np.clip(base + rng.normal(0, 0.06, base.shape), 1e-3, None)
    p /= p.sum(axis=1, keepdims=True)
    return pl.DataFrame({"match_id": ids, "home_win_prob": p[:, 0],
                         "draw_prob": p[:, 1], "away_win_prob": p[:, 2]}), False


market, real_mkt = _market(remaining["match_id"].to_list(), remaining)
live_edges = compute_edges(remaining, market, threshold=0.05)
print(f"Market: {'REAL' if real_mkt else 'SYNTHETIC'} | {live_edges.height} live edges on remaining matches")
live_edges.head(8)

## Model staleness (spec 4.5)

`StalenessMonitor` accumulates per-match surprise (`|actual - predicted|` for the realized
outcome). Crossing the yellow threshold (75th percentile of historical matchday sequences)
is advisory; crossing red (90th) triggers a full Bayesian re-estimation. We feed each
completed match's pre-match predicted probability for its realized outcome. Thresholds here
are illustrative — production calibrates them from historical tournaments.

In [ ]:
_OUTCOME_IDX = {"home": 0, "draw": 1, "away": 2}
# Illustrative thresholds; real values come from historical matchday-surprise percentiles.
monitor = StalenessMonitor(yellow=0.40 * results.height, red=0.55 * results.height)

trace = []
for r in results.iter_rows(named=True):
    probs = _match_hda(base_model, r["home_team"], r["away_team"])
    if r["home_goals"] > r["away_goals"]:
        realized = "home"
    elif r["home_goals"] < r["away_goals"]:
        realized = "away"
    else:
        realized = "draw"
    p_realized = probs[_OUTCOME_IDX[realized]]
    level = monitor.observe(p_realized, occurred=True)
    trace.append({
        "match_id": r["match_id"], "realized": realized,
        "p_predicted": round(p_realized, 3),
        "surprise": round(surprise_increment(p_realized, True), 3),
        "cumulative": round(monitor.cumulative, 3), "level": level,
    })

staleness = pl.DataFrame(trace)
print(f"Final staleness level: {monitor.level.upper()} "
      f"(cumulative surprise {monitor.cumulative:.2f} over {monitor.n} matches)")
staleness

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, staleness.height + 1), staleness["cumulative"], "o-", color="slateblue")
ax.axhline(monitor.yellow, color="goldenrod", ls="--", label="yellow (advisory)")
ax.axhline(monitor.red, color="crimson", ls="--", label="red (re-estimate)")
ax.set_xlabel("matches observed")
ax.set_ylabel("cumulative surprise")
ax.set_title("Model staleness tracking (spec 4.5)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

- Live results drive a lightweight strength re-estimation and a bracket re-simulation
  (spec 4.4); trophy-probability shifts > 0.5pp are flagged like the agent's Reflect node
  (spec 5.2).
- Market edges are recomputed on the remaining fixtures so the edge table tracks the updated
  model (spec 3.6).
- `StalenessMonitor` accumulates forecast surprise and reports green/yellow/red against the
  spec 4.5 intervention thresholds.

Wired to real ingested live results and trained artifacts, this is the data backing the
Streamlit dashboard's live pages; raise `N_SIMS` to 100,000 for production refreshes.